# McByte++ — Melbourne Victory v Auckland FC, 17 May 2025
Select a GPU runtime, then run cells in order. Upload your local 11.3-second `MV_Clip.mp4` when prompted. Model receives numbered JPEG frames, then annotated frames are assembled into an MP4.

This notebook uses a separate Python 3.10 environment. Every installation and model command targets that environment explicitly; the Colab kernel stays unchanged. Failed commands display their logs and stop the cell.

Tracking rate defaults to 10 fps. Matching the SkillCorner sample rate does not establish time alignment: video-to-match timing and player identity mapping come later. Visual track IDs are not SkillCorner player IDs.

Custom Colab notebook used for this project, based on the [official McByte++ installation instructions](https://github.com/tstanczyk95/McBytePlusPlus/blob/main/INSTALLATION.md). GPU inference completed successfully for this clip in Colab. Results may vary with runtime hardware and dependency versions.

## 1. Configuration and GPU

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil

REPO = Path('/content/McBytePlusPlus')
PY = Path('/content/mcbyte-venv/bin/python')
FPS = 10
EXPERIMENT = 'victory_auckland_leg1'
MATCH_ID = '2017461'

# Repair a PATH left by an earlier notebook version without dropping valid entries.
entries = [p for p in os.environ.get('PATH', '').split(':') if p and '$' not in p]
entries += ['/usr/local/bin', '/usr/bin', '/bin', '/usr/local/sbin', '/usr/sbin', '/sbin']
os.environ['PATH'] = ':'.join(dict.fromkeys(entries))

def run(args, cwd=None, log_path=None):
    args = [str(a) for a in args]
    print('Running:', ' '.join(args), flush=True)
    # Colab's inline backend is unavailable in the separate model environment.
    child_env = os.environ.copy()
    child_env['MPLBACKEND'] = 'Agg'
    log = open(log_path, 'w') if log_path else None
    try:
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1, env=child_env) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if log:
                    log.write(line)
                    log.flush()
            status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, args)
    finally:
        if log:
            log.close()

assert shutil.which('nvidia-smi'), 'Select a GPU runtime first.'
run(['nvidia-smi'])
print('Colab kernel:', sys.version)

## 2. Python environment and main tracker installation

In [ ]:
if not REPO.exists():
    run(['git', 'clone', '--recursive', 'https://github.com/tstanczyk95/McBytePlusPlus.git', REPO])
assert (REPO / 'tools/demo_track__with_reid.py').is_file(), 'Repository missing or incomplete.'
run(['git', 'submodule', 'update', '--init', '--recursive'], cwd=REPO)
run([sys.executable, '-m', 'pip', 'install', 'uv'])
uv = [sys.executable, '-m', 'uv']
if not PY.exists():
    run(uv + ['venv', '--seed', '--python', '3.10', PY.parent.parent])
run(uv + ['pip', 'install', '--python', PY, 'pip', 'setuptools', 'wheel'])
run([PY, '--version'])

constraints = Path('/content/mcbyte_constraints.txt')
constraints.write_text('numpy==1.24.4\nopencv-python==4.10.0.84\ntorch==2.9.1\ntorchvision==0.24.1\ntorchaudio==2.9.1\n')

def install(*args, cwd=REPO):
    run([PY, '-m', 'pip', 'install', '-c', constraints, *args], cwd=cwd)

install('torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1',
        '--index-url', 'https://download.pytorch.org/whl/cu128')
install('numpy==1.24.4', 'opencv-python==4.10.0.84', 'cython', 'ninja')
install('-r', str(REPO / 'requirements.txt'))
install('-e', str(REPO), '--no-build-isolation')
install('git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI', '--no-build-isolation')
install('cython_bbox', '--no-build-isolation')
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', 'ffmpeg'])
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('McByte++ revision:', revision)

## 3. EdgeTAM, re-ID, and environment checks

In [ ]:
edge = REPO / 'mask_propagation/EdgeTAM'
reid = REPO / 'deep-person-reid'
assert (edge / 'setup.py').exists(), 'EdgeTAM source missing.'
assert (reid / 'setup.py').exists(), 'Re-ID source missing.'
install('-e', str(edge), '--no-build-isolation', cwd=edge)
install('-r', str(reid / 'requirements.txt'), cwd=reid)
install('setuptools==59.8.0', cwd=reid)
run([PY, 'setup.py', 'develop'], cwd=reid)
run([PY, '-m', 'pip', 'check'])
run([PY, '-c', "import torch, torchvision, cv2, numpy, yolox, sam2, torchreid, cython_bbox, lap; assert torch.cuda.is_available(), 'CUDA unavailable in McByte environment'; print('GPU:', torch.cuda.get_device_name(0)); print('OpenCV:', cv2.__version__, 'NumPy:', numpy.__version__); print('BF16 supported:', torch.cuda.is_bf16_supported())"], cwd=REPO)

## 4. Download pretrained weights

In [ ]:
install('gdown')
weights_dir = Path('/content/mcbyte_weights')
weights = {
    'yolox_x_sports_mix.pth.tar': REPO / 'pretrained/yolox_x_sports_mix.pth.tar',
    'edgetam.pt': REPO / 'mask_propagation/EdgeTAM/checkpoints/edgetam.pt',
    'sports_model.pth.tar-60': REPO / 'deep-person-reid/pretrained/sports_model.pth.tar-60',
}
def payload_ok(path):
    return path.is_file() and path.stat().st_size > 100_000

if not all(payload_ok(p) for p in weights.values()):
    # Backup folder linked directly by the McByte++ installation guide.
    run([PY, '-m', 'gdown', '--folder',
         'https://drive.google.com/drive/folders/1yzzJk9dpJUY3lIHdkkFyGtKL2F-FenN6',
         '-O', weights_dir])
    for filename, destination in weights.items():
        if payload_ok(destination):
            continue
        candidates = [p for p in weights_dir.rglob(filename) if payload_ok(p)]
        if not candidates:
            raise FileNotFoundError(f'{filename} was not downloaded. Inspect download output above.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidates[0], destination)
for destination in weights.values():
    assert payload_ok(destination), f'Missing checkpoint: {destination}'
    print(destination.name, round(destination.stat().st_size / 1024**2, 1), 'MB')

## 5. Upload MV_Clip.mp4

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload one MP4 file.')
filename, data = next(iter(uploaded.items()))
video_file = Path('/content') / Path(filename).name
if video_file.suffix.lower() != '.mp4':
    raise ValueError('Select MV_Clip.mp4.')
video_file.write_bytes(data)
video_path = str(video_file)
probe = subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries',
    'format=duration:stream=codec_type,width,height,avg_frame_rate', '-of', 'json', video_path], text=True)
video_info = json.loads(probe)
print('Saved:', video_path)
print(json.dumps(video_info, indent=2))

## 6. Extract numbered JPEG frames

In [ ]:
# EdgeTAM requires integer filenames: 000001.jpg, not frame_000001.jpg.
# A new folder per extraction prevents leftover frames from older clips.
import tempfile
frames_dir = Path(tempfile.mkdtemp(prefix='mv_frames_', dir='/content'))
run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', video_path,
     '-vf', f'fps={FPS}', '-q:v', '2', '-start_number', '1', frames_dir / '%06d.jpg'])
frame_files = sorted(frames_dir.glob('*.jpg'))
assert frame_files, 'Frame extraction produced no frames.'
assert all(p.stem.isdigit() for p in frame_files)
print('Frames:', len(frame_files), '| FPS:', FPS, '| Folder:', frames_dir)
from IPython.display import display, Image
display(Image(filename=str(frame_files[0])))

## 7. Colab compatibility adjustments

In [ ]:
# Remove GUI waits from both demo scripts; Colab has no interactive OpenCV window.
for name in ['demo_track__with_reid.py', 'demo_track__no_reid.py']:
    script = REPO / 'tools' / name
    source = script.read_text()
    if 'ch = cv2.waitKey(0)' not in source and 'ch = -1' not in source:
        raise RuntimeError(f'{name} changed upstream. Check its GUI wait before proceeding.')
    script.write_text(source.replace('ch = cv2.waitKey(0)', 'ch = -1'))

# BF16 autocast is unsupported on some GPUs (e.g. T4).
# Disable that autocast on such GPUs; EdgeTAM runs in normal precision there.
manager = REPO / 'mask_propagation/mask_manager__edgetam.py'
source = manager.read_text()
old_autocast = 'torch.autocast("cuda", dtype=torch.bfloat16)'
new_autocast = 'torch.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_bf16_supported())'
if old_autocast not in source and new_autocast not in source:
    raise RuntimeError('Mask manager changed upstream. Check precision handling.')
source = source.replace(old_autocast, new_autocast)
source = source.replace('self.predictor.init_state(video_path=input_frame_dir)',
                        'self.predictor.init_state(video_path=input_frame_dir, offload_video_to_cpu=True)')
manager.write_text(source)
print('GUI waits removed; GPU precision fallback enabled; input frames stored in CPU memory.')

## 8. TRACKER — run McByte++

In [ ]:
import datetime
# A failed rerun must not leave previous outputs eligible for export.
for name in ['run_dir', 'mot_path']:
    globals().pop(name, None)
run_name = EXPERIMENT + '_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
output_root = REPO / 'YOLOX_outputs' / run_name / 'track_vis'
log_path = Path('/content/MV_Clip_tracker.log')
run([PY, 'tools/demo_track__with_reid.py', '--path', frames_dir, '--fps', FPS,
     '--vis_type', 'basic', '--experiment-name', run_name, '--reid_model_path',
     weights['sports_model.pth.tar-60']], cwd=REPO, log_path=log_path)
completed = [p for p in output_root.glob('*') if p.is_dir()
             and (p.parent / (p.name + '.txt')).is_file()
             and (p / '000001.jpg').is_file()]
if not completed:
    raise RuntimeError(f'Tracker exited without expected output. Inspect {log_path}.')
run_dir = max(completed, key=lambda p: p.stat().st_mtime)
mot_path = run_dir.parent / (run_dir.name + '.txt')
assert len(list(run_dir.glob('*.jpg'))) == len(frame_files), 'Annotated frame count mismatch.'
print('Tracking complete:', run_dir)

## 9. Render MP4, export tracks, and download

In [ ]:
assert 'run_dir' in globals() and run_dir.is_dir(), 'Run TRACKER cell successfully first.'
import csv
tracks_path = Path('/content/MV_Clip_tracks.csv')
rows = list(csv.reader(mot_path.open()))
if not rows:
    raise RuntimeError('Tracker produced no tracks. Inspect tracker log.')
with tracks_path.open('w', newline='') as out:
    writer = csv.writer(out)
    writer.writerow(['frame', 'track_id', 'x', 'y', 'width', 'height', 'score',
                     'mot_x', 'mot_y', 'mot_z', 'clip_time_seconds'])
    for row in rows:
        if len(row) != 10:
            raise ValueError(f'Unexpected MOT row: {row}')
        writer.writerow(row + [round((int(row[0]) - 1) / FPS, 6)])
annotated_path = Path('/content/MV_Clip_mcbyte.mp4')
run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-framerate', FPS,
     '-start_number', '1', '-i', run_dir / '%06d.jpg', '-i', video_path,
     '-map', '0:v:0', '-map', '1:a?', '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
     '-c:a', 'aac', '-shortest', annotated_path])
assert annotated_path.is_file() and annotated_path.stat().st_size > 0
metadata_path = Path('/content/MV_Clip_metadata.json')
metadata_path.write_text(json.dumps({'match_id': MATCH_ID, 'source_video': video_file.name,
    'source_info': video_info, 'tracking_fps': FPS, 'frame_count': len(frame_files),
    'mcbyte_revision': revision, 'mcbyte_run': run_name,
    'match_clock_offset_seconds': None, 'player_id_mapping': None}, indent=2))
from IPython.display import Video
display(Video(str(annotated_path), embed=True))
print('Track rows:', len(rows))
from google.colab import files
for artifact in [annotated_path, tracks_path, metadata_path]:
    files.download(str(artifact))